# Basic Bearing Predictive Maintenance Model Using Run-to-Failure Dataset

This notebook builds a memory-safe baseline anomaly model from run-to-failure bearing data.

Key design choices:
- Process files one by one
- Use chunk-based reading for CSV/TXT/DAT files
- Keep only current-file features in memory before appending to disk
- Save all outputs under /kaggle/working

## 1. Setup

In [ ]:
import os
import gc
import json
import re
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import scipy
import sklearn
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from scipy.io import loadmat
from scipy.stats import skew, kurtosis
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

DATASET_ROOT = Path("/kaggle/input/datasets/ruththra/bearing-predictive-maintenance-logs")
WORKING_DIR = Path("/kaggle/working")
PROCESSED_DIR = WORKING_DIR / "processed"
MODEL_DIR = WORKING_DIR / "models"
REPORT_DIR = WORKING_DIR / "reports"

for directory in [PROCESSED_DIR, MODEL_DIR, REPORT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Start with a small subset for validation in Kaggle, then increase as needed.
MAX_FILES = None
WINDOW_SIZE = 4096
OVERLAP = 0.5
RANDOM_STATE = 42

STEP_SIZE = max(1, int(WINDOW_SIZE * (1 - OVERLAP)))
CHUNK_ROWS = 200_000

FILE_INDEX_PATH = PROCESSED_DIR / "file_index.csv"
FEATURES_PATH = PROCESSED_DIR / "bearing_features.csv"
LABELED_FEATURES_PATH = PROCESSED_DIR / "bearing_features_labeled.csv"

MODEL_PATH = MODEL_DIR / "vibration_isolation_forest.pkl"
FEATURE_COLUMNS_PATH = MODEL_DIR / "feature_columns.json"
THRESHOLDS_PATH = MODEL_DIR / "anomaly_thresholds.json"

SKIPPED_FILES_PATH = REPORT_DIR / "skipped_files.csv"
FEATURE_EXTRACTION_REPORT_PATH = REPORT_DIR / "feature_extraction_report.txt"
EVAL_REPORT_PATH = REPORT_DIR / "model_evaluation_report.txt"
CONFUSION_MATRIX_PATH = REPORT_DIR / "confusion_matrix.csv"

pd.options.display.max_columns = 500
plt.style.use("seaborn-v0_8-whitegrid")

print("Setup complete.")
print(f"Dataset root: {DATASET_ROOT}")
print(f"Working directory: {WORKING_DIR}")
print(f"MAX_FILES: {MAX_FILES}")
print(f"Current working directory (os): {os.getcwd()}")
print(f"scipy version: {scipy.__version__}")
print(f"sklearn version: {sklearn.__version__}")

Setup complete.
Dataset root: /kaggle/input/datasets/ruththra/bearing-predictive-maintenance-logs
Working directory: /kaggle/working
MAX_FILES: None
Current working directory (os): /kaggle/working
scipy version: 1.16.3
sklearn version: 1.6.1


## 2. Dataset Inspection

In [14]:
SUPPORTED_EXTENSIONS = {".csv", ".txt", ".dat", ".xlsx", ".mat"}

file_records = []
for file_path in DATASET_ROOT.rglob("*"):
    if file_path.is_file() and file_path.suffix.lower() in SUPPORTED_EXTENSIONS:
        file_records.append({
            "file_path": str(file_path),
            "file_name": file_path.name,
            "extension": file_path.suffix.lower(),
            "parent_folder": file_path.parent.name,
            "file_size_mb": file_path.stat().st_size / (1024 ** 2)
        })

file_index_df = pd.DataFrame(file_records)
if not file_index_df.empty:
    file_index_df = file_index_df.sort_values("file_path").reset_index(drop=True)

file_index_df.to_csv(FILE_INDEX_PATH, index=False)

display(file_index_df.head())
print(f"Total supported files discovered: {len(file_index_df)}")
if not file_index_df.empty:
    print(f"Total supported dataset size (MB): {file_index_df['file_size_mb'].sum():,.2f}")
else:
    print("Total supported dataset size (MB): 0.00")
print(f"Saved file index to: {FILE_INDEX_PATH}")

,file_path,file_name,extension,parent_folder,file_size_mb
0,/kaggle/input/datasets/ruththra/bearing-predic...,LogFile_2022-06-20-17-00-31.csv,.csv,bearing-predictive-maintenance-logs,136.913760
1,/kaggle/input/datasets/ruththra/bearing-predic...,LogFile_2022-06-20-18-00-31.csv,.csv,bearing-predictive-maintenance-logs,137.071884
2,/kaggle/input/datasets/ruththra/bearing-predic...,LogFile_2022-06-20-19-00-31.csv,.csv,bearing-predictive-maintenance-logs,137.103577
3,/kaggle/input/datasets/ruththra/bearing-predic...,LogFile_2022-06-20-20-00-31.csv,.csv,bearing-predictive-maintenance-logs,136.944926
4,/kaggle/input/datasets/ruththra/bearing-predic...,LogFile_2022-06-20-21-00-31.csv,.csv,bearing-predictive-maintenance-logs,136.938971


Total supported files discovered: 129
Total supported dataset size (MB): 17,675.88
Saved file index to: /kaggle/working/processed/file_index.csv


## 3. Safe Numeric File Reader

In [ ]:
def detect_numeric_columns(df):
    """Return columns that contain at least some numeric values."""
    if df is None or df.empty:
        return []

    numeric_df = df.apply(pd.to_numeric, errors="coerce")
    numeric_cols = [col for col in numeric_df.columns if numeric_df[col].notna().any()]
    return numeric_cols


def normalize_column_name(name):
    text = str(name).strip().lower()
    text = re.sub(r"[\[\](){}]", " ", text)
    text = text.replace("-", " ").replace("/", " ")
    text = re.sub(r"[^a-z0-9]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")
    return text


def normalize_dataset_columns(df):
    """
    Map known dataset names to standard names while being robust to
    case differences, spaces, brackets, hyphens, and underscores.
    """
    if df is None or df.empty:
        return df

    alias_map = {
        "vibration_x": {
            "vibration_x",
            "vibration_x_axis",
            "vibration_axis_x",
            "vibrationx",
            "vibrationxaxis",
            "vibration_xaxis",
            "vibration_x_axis_signal",
        },
        "vibration_y": {
            "vibration_y",
            "vibration_y_axis",
            "vibration_axis_y",
            "vibrationy",
            "vibrationyaxis",
            "vibration_yaxis",
            "vibration_y_axis_signal",
        },
        "temperature_bearing": {
            "temperature_bearing",
            "bearing_temperature",
            "temperature_bearing_sensor",
            "temperaturebearing",
            "temp_bearing",
            "bearing_temp",
        },
        "temperature_atmospheric": {
            "temperature_atmospheric",
            "atmospheric_temperature",
            "temperature_atmosphere",
            "temperature_ambient",
            "ambient_temperature",
            "temperatureatmospheric",
            "temp_atmospheric",
            "atmospheric_temp",
            "temperature_air",
            "air_temperature",
        },
    }

    rename_map = {}
    for col in df.columns:
        normalized = normalize_column_name(col)
        for target, aliases in alias_map.items():
            if normalized == target or normalized in aliases:
                rename_map[col] = target
                break

    if rename_map:
        df = df.rename(columns=rename_map)

    return df


def infer_sensor_columns(df):
    """Infer required sensor columns using normalized names, then numeric fallback."""
    columns = list(df.columns)
    numeric_cols = detect_numeric_columns(df)

    sensor_cols = {
        "vibration_x": "vibration_x" if "vibration_x" in columns else None,
        "vibration_y": "vibration_y" if "vibration_y" in columns else None,
        "temperature_bearing": "temperature_bearing" if "temperature_bearing" in columns else None,
        "temperature_atmospheric": "temperature_atmospheric" if "temperature_atmospheric" in columns else None,
    }

    used = set(col for col in sensor_cols.values() if col is not None)
    fallback_numeric = [col for col in numeric_cols if col not in used]

    if sensor_cols["vibration_x"] is None and fallback_numeric:
        sensor_cols["vibration_x"] = fallback_numeric.pop(0)
    if sensor_cols["vibration_y"] is None and fallback_numeric:
        sensor_cols["vibration_y"] = fallback_numeric.pop(0)
    if sensor_cols["temperature_bearing"] is None and fallback_numeric:
        sensor_cols["temperature_bearing"] = fallback_numeric.pop(0)
    if sensor_cols["temperature_atmospheric"] is None and fallback_numeric:
        sensor_cols["temperature_atmospheric"] = fallback_numeric.pop(0)

    return sensor_cols


def read_csv_txt_dat_safely(file_path):
    """
    Read CSV/TXT/DAT with chunking while preserving headers when present.
    Falls back to header-less parsing and numeric column detection when needed.
    """
    read_attempts = [
        {"sep": ",", "engine": "c"},
        {"sep": ";", "engine": "python"},
        {"sep": r"\s+", "engine": "python"},
    ]

    parse_modes = [
        {"header": 0, "tag": "with_header"},
        {"header": None, "tag": "no_header"},
    ]

    last_error = "Could not parse file with supported separators and header modes."

    for attempt in read_attempts:
        for mode in parse_modes:
            try:
                preview_df = pd.read_csv(
                    file_path,
                    sep=attempt["sep"],
                    engine=attempt["engine"],
                    header=mode["header"],
                    nrows=5000,
                    on_bad_lines="skip",
                )

                if preview_df is None or preview_df.empty:
                    continue

                preview_df = normalize_dataset_columns(preview_df)
                numeric_cols = detect_numeric_columns(preview_df)
                if not numeric_cols:
                    continue

                def chunk_iterator(read_options=attempt.copy(), parse_mode=mode.copy()):
                    for chunk in pd.read_csv(
                        file_path,
                        sep=read_options["sep"],
                        engine=read_options["engine"],
                        header=parse_mode["header"],
                        chunksize=CHUNK_ROWS,
                        on_bad_lines="skip",
                    ):
                        if chunk is None or chunk.empty:
                            continue
                        chunk = normalize_dataset_columns(chunk)
                        yield chunk

                return chunk_iterator(), None

            except Exception as exc:
                last_error = f"{type(exc).__name__}: {exc}"

    return None, last_error


def read_xlsx_safely(file_path):
    """Read XLSX safely and return one DataFrame iterator."""
    try:
        df = pd.read_excel(file_path)
    except Exception as exc:
        return None, f"XLSX read error: {exc}"

    if df is None or df.empty:
        return None, "XLSX file is empty."

    df = normalize_dataset_columns(df)
    numeric_cols = detect_numeric_columns(df)
    if not numeric_cols:
        return None, "No numeric columns found in XLSX file."

    def xlsx_iterator():
        yield df

    return xlsx_iterator(), None


def read_mat_safely(file_path):
    """Read MAT file safely and return one numeric DataFrame as an iterator."""
    try:
        mat_data = loadmat(file_path)
    except Exception as exc:
        return None, f"MAT read error: {exc}"

    numeric_arrays = {}
    for key, value in mat_data.items():
        if key.startswith("__"):
            continue

        arr = np.asarray(value)
        if np.issubdtype(arr.dtype, np.number):
            flat = arr.astype(float).ravel()
            if flat.size > 0:
                numeric_arrays[key] = flat

    if not numeric_arrays:
        return None, "No numeric arrays found in MAT file."

    max_len = max(len(arr) for arr in numeric_arrays.values())
    aligned = {}
    for key, arr in numeric_arrays.items():
        if len(arr) < max_len:
            padded = np.full(max_len, np.nan, dtype=float)
            padded[:len(arr)] = arr
            aligned[key] = padded
        else:
            aligned[key] = arr

    mat_df = pd.DataFrame(aligned)
    mat_df = normalize_dataset_columns(mat_df)

    def mat_iterator():
        yield mat_df

    return mat_iterator(), None


def get_chunk_iterator(file_path):
    suffix = Path(file_path).suffix.lower()

    if suffix in {".csv", ".txt", ".dat"}:
        return read_csv_txt_dat_safely(file_path)
    if suffix == ".xlsx":
        return read_xlsx_safely(file_path)
    if suffix == ".mat":
        return read_mat_safely(file_path)

    return None, f"Unsupported extension: {suffix}"

## 4. Feature Extraction Functions

In [ ]:
VIBRATION_METRIC_NAMES = [
    "mean",
    "standard_deviation",
    "minimum",
    "maximum",
    "peak_to_peak",
    "rms",
    "skewness",
    "kurtosis",
    "crest_factor",
    "energy",
    "dominant_frequency_index",
    "spectral_energy",
    "spectral_centroid_index",
]

TEMPERATURE_FEATURE_COLUMNS = [
    "temperature_bearing_mean",
    "temperature_bearing_min",
    "temperature_bearing_max",
    "temperature_bearing_std",
    "temperature_bearing_trend",
    "temperature_atmospheric_mean",
    "temperature_atmospheric_min",
    "temperature_atmospheric_max",
    "temperature_atmospheric_std",
    "temperature_difference_mean",
    "temperature_difference_max",
    "temperature_difference_trend",
]


def empty_temperature_features():
    return {key: np.nan for key in TEMPERATURE_FEATURE_COLUMNS}


def compute_linear_trend(signal):
    signal = np.asarray(signal, dtype=float)
    signal = signal[np.isfinite(signal)]
    if signal.size <= 1:
        return 0.0
    x = np.arange(signal.size, dtype=float)
    return float(np.polyfit(x, signal, 1)[0])


def compute_signal_features(window, prefix):
    """Compute prefixed vibration features for one signal window."""
    signal = np.asarray(window, dtype=float)
    signal = signal[np.isfinite(signal)]

    if signal.size == 0:
        return {f"{prefix}_{metric}": np.nan for metric in VIBRATION_METRIC_NAMES}

    mean_val = float(np.mean(signal))
    std_val = float(np.std(signal))
    min_val = float(np.min(signal))
    max_val = float(np.max(signal))
    ptp_val = float(np.ptp(signal))
    rms_val = float(np.sqrt(np.mean(signal ** 2)))

    skew_val = float(skew(signal, bias=False)) if signal.size > 2 else 0.0
    kurt_val = float(kurtosis(signal, bias=False, fisher=False)) if signal.size > 3 else 0.0

    if not np.isfinite(skew_val):
        skew_val = 0.0
    if not np.isfinite(kurt_val):
        kurt_val = 0.0

    crest_factor_val = float(np.max(np.abs(signal)) / (rms_val + 1e-12))
    energy_val = float(np.sum(signal ** 2))

    centered = signal - mean_val
    spectrum = np.abs(np.fft.rfft(centered))
    power = spectrum ** 2

    dominant_frequency_index_val = int(np.argmax(power)) if power.size > 0 else 0
    spectral_energy_val = float(np.sum(power))

    if spectral_energy_val > 0 and power.size > 0:
        freq_idx = np.arange(power.size, dtype=float)
        spectral_centroid_index_val = float(np.sum(freq_idx * power) / spectral_energy_val)
    else:
        spectral_centroid_index_val = 0.0

    values = {
        "mean": mean_val,
        "standard_deviation": std_val,
        "minimum": min_val,
        "maximum": max_val,
        "peak_to_peak": ptp_val,
        "rms": rms_val,
        "skewness": skew_val,
        "kurtosis": kurt_val,
        "crest_factor": crest_factor_val,
        "energy": energy_val,
        "dominant_frequency_index": dominant_frequency_index_val,
        "spectral_energy": spectral_energy_val,
        "spectral_centroid_index": spectral_centroid_index_val,
    }

    return {f"{prefix}_{key}": value for key, value in values.items()}


def compute_temperature_features(bearing_window, atmospheric_window):
    """
    Compute bearing, atmospheric, and differential temperature features.
    Bearing is treated as the primary machine temperature signal.
    """
    if bearing_window is None or atmospheric_window is None:
        return empty_temperature_features()

    bearing = np.asarray(bearing_window, dtype=float)
    atmospheric = np.asarray(atmospheric_window, dtype=float)

    bearing_finite = bearing[np.isfinite(bearing)]
    atmospheric_finite = atmospheric[np.isfinite(atmospheric)]

    if bearing_finite.size == 0 and atmospheric_finite.size == 0:
        return empty_temperature_features()

    aligned_mask = np.isfinite(bearing) & np.isfinite(atmospheric)
    if aligned_mask.any():
        difference = bearing[aligned_mask] - atmospheric[aligned_mask]
    else:
        difference = np.array([], dtype=float)

    features = {
        "temperature_bearing_mean": float(np.mean(bearing_finite)) if bearing_finite.size else np.nan,
        "temperature_bearing_min": float(np.min(bearing_finite)) if bearing_finite.size else np.nan,
        "temperature_bearing_max": float(np.max(bearing_finite)) if bearing_finite.size else np.nan,
        "temperature_bearing_std": float(np.std(bearing_finite)) if bearing_finite.size else np.nan,
        "temperature_bearing_trend": compute_linear_trend(bearing_finite),
        "temperature_atmospheric_mean": float(np.mean(atmospheric_finite)) if atmospheric_finite.size else np.nan,
        "temperature_atmospheric_min": float(np.min(atmospheric_finite)) if atmospheric_finite.size else np.nan,
        "temperature_atmospheric_max": float(np.max(atmospheric_finite)) if atmospheric_finite.size else np.nan,
        "temperature_atmospheric_std": float(np.std(atmospheric_finite)) if atmospheric_finite.size else np.nan,
        "temperature_difference_mean": float(np.mean(difference)) if difference.size else np.nan,
        "temperature_difference_max": float(np.max(difference)) if difference.size else np.nan,
        "temperature_difference_trend": compute_linear_trend(difference),
    }

    return features

## 5. Memory-Safe Feature Extraction

In [ ]:
def extract_features_from_single_file(file_path, global_start_index):
    """Extract windowed features from one file and return next global index and detection info."""
    chunk_iterator, read_error = get_chunk_iterator(file_path)
    if chunk_iterator is None:
        return pd.DataFrame(), global_start_index, read_error, {}

    file_feature_rows = []
    file_window_index = 0
    global_order_index = global_start_index

    sensor_columns = None
    detection_flags = {
        "vibration_x_detected": False,
        "vibration_y_detected": False,
        "temperature_bearing_detected": False,
        "temperature_atmospheric_detected": False,
    }

    buffer_x = np.array([], dtype=float)
    buffer_y = np.array([], dtype=float)
    buffer_temp_bearing = np.array([], dtype=float)
    buffer_temp_atmospheric = np.array([], dtype=float)

    for chunk_df in chunk_iterator:
        if chunk_df is None or chunk_df.empty:
            continue

        chunk_df = normalize_dataset_columns(chunk_df)

        if sensor_columns is None:
            sensor_columns = infer_sensor_columns(chunk_df)

            detection_flags["vibration_x_detected"] = sensor_columns.get("vibration_x") is not None
            detection_flags["vibration_y_detected"] = sensor_columns.get("vibration_y") is not None
            detection_flags["temperature_bearing_detected"] = sensor_columns.get("temperature_bearing") is not None
            detection_flags["temperature_atmospheric_detected"] = sensor_columns.get("temperature_atmospheric") is not None

            if sensor_columns["vibration_x"] is None or sensor_columns["vibration_y"] is None:
                return pd.DataFrame(), global_start_index, "Could not detect both vibration axes.", detection_flags

        vx_col = sensor_columns["vibration_x"]
        vy_col = sensor_columns["vibration_y"]
        tb_col = sensor_columns["temperature_bearing"]
        ta_col = sensor_columns["temperature_atmospheric"]

        if vx_col not in chunk_df.columns or vy_col not in chunk_df.columns:
            continue

        vx_series = pd.to_numeric(chunk_df[vx_col], errors="coerce")
        vy_series = pd.to_numeric(chunk_df[vy_col], errors="coerce")

        valid_vibration_mask = vx_series.notna().to_numpy() & vy_series.notna().to_numpy()
        if valid_vibration_mask.sum() == 0:
            continue

        vx_values = vx_series.to_numpy(dtype=float)[valid_vibration_mask]
        vy_values = vy_series.to_numpy(dtype=float)[valid_vibration_mask]

        if tb_col is not None and tb_col in chunk_df.columns:
            tb_series = pd.to_numeric(chunk_df[tb_col], errors="coerce").to_numpy(dtype=float)
            tb_values = tb_series[valid_vibration_mask]
        else:
            tb_values = np.full(vx_values.shape[0], np.nan, dtype=float)

        if ta_col is not None and ta_col in chunk_df.columns:
            ta_series = pd.to_numeric(chunk_df[ta_col], errors="coerce").to_numpy(dtype=float)
            ta_values = ta_series[valid_vibration_mask]
        else:
            ta_values = np.full(vx_values.shape[0], np.nan, dtype=float)

        buffer_x = np.concatenate([buffer_x, vx_values])
        buffer_y = np.concatenate([buffer_y, vy_values])
        buffer_temp_bearing = np.concatenate([buffer_temp_bearing, tb_values])
        buffer_temp_atmospheric = np.concatenate([buffer_temp_atmospheric, ta_values])

        while buffer_x.size >= WINDOW_SIZE and buffer_y.size >= WINDOW_SIZE:
            window_x = buffer_x[:WINDOW_SIZE]
            window_y = buffer_y[:WINDOW_SIZE]
            window_resultant = np.sqrt((window_x ** 2) + (window_y ** 2))

            window_temp_bearing = buffer_temp_bearing[:WINDOW_SIZE] if buffer_temp_bearing.size >= WINDOW_SIZE else None
            window_temp_atmospheric = buffer_temp_atmospheric[:WINDOW_SIZE] if buffer_temp_atmospheric.size >= WINDOW_SIZE else None

            row = {}
            row.update(compute_signal_features(window_x, "vibration_x"))
            row.update(compute_signal_features(window_y, "vibration_y"))
            row.update(compute_signal_features(window_resultant, "vibration_resultant"))
            row.update(compute_temperature_features(window_temp_bearing, window_temp_atmospheric))

            row["source_file"] = str(file_path)
            row["window_index"] = int(file_window_index)
            row["global_order_index"] = int(global_order_index)
            file_feature_rows.append(row)

            file_window_index += 1
            global_order_index += 1

            buffer_x = buffer_x[STEP_SIZE:]
            buffer_y = buffer_y[STEP_SIZE:]
            buffer_temp_bearing = buffer_temp_bearing[STEP_SIZE:]
            buffer_temp_atmospheric = buffer_temp_atmospheric[STEP_SIZE:]

    if len(file_feature_rows) == 0:
        return pd.DataFrame(), global_start_index, "No complete windows extracted.", detection_flags

    return pd.DataFrame(file_feature_rows), global_order_index, None, detection_flags


file_index_df = pd.read_csv(FILE_INDEX_PATH) if FILE_INDEX_PATH.exists() else pd.DataFrame()

if FEATURES_PATH.exists():
    FEATURES_PATH.unlink()

EXTRACTION_SUMMARY = {
    "number_of_files_discovered": int(len(file_index_df)),
    "number_of_files_selected": 0,
    "number_of_files_processed": 0,
    "number_of_skipped_files": 0,
    "total_extracted_windows": 0,
    "vibration_x_detected": False,
    "vibration_y_detected": False,
    "temperature_bearing_detected": False,
    "temperature_atmospheric_detected": False,
    "final_feature_column_count": 0,
}

# Initialize skipped report with headers so read_csv never fails with EmptyDataError.
skipped_df = pd.DataFrame(columns=["file_path", "reason"])
skipped_df.to_csv(SKIPPED_FILES_PATH, index=False)

if file_index_df.empty:
    print("No supported files found. Stop after dataset inspection.")
else:
    files_to_process = file_index_df.head(MAX_FILES).copy()
    EXTRACTION_SUMMARY["number_of_files_selected"] = int(len(files_to_process))

    skipped_records = []
    processed_files = 0
    global_idx = 0
    wrote_header = False

    for _, file_row in tqdm(files_to_process.iterrows(), total=len(files_to_process), desc="Feature extraction"):
        file_path = Path(file_row["file_path"])

        try:
            file_features_df, global_idx, error_msg, detection_flags = extract_features_from_single_file(file_path, global_idx)

            EXTRACTION_SUMMARY["vibration_x_detected"] = EXTRACTION_SUMMARY["vibration_x_detected"] or detection_flags.get("vibration_x_detected", False)
            EXTRACTION_SUMMARY["vibration_y_detected"] = EXTRACTION_SUMMARY["vibration_y_detected"] or detection_flags.get("vibration_y_detected", False)
            EXTRACTION_SUMMARY["temperature_bearing_detected"] = EXTRACTION_SUMMARY["temperature_bearing_detected"] or detection_flags.get("temperature_bearing_detected", False)
            EXTRACTION_SUMMARY["temperature_atmospheric_detected"] = EXTRACTION_SUMMARY["temperature_atmospheric_detected"] or detection_flags.get("temperature_atmospheric_detected", False)

            if file_features_df.empty:
                skipped_records.append({
                    "file_path": str(file_path),
                    "reason": error_msg if error_msg else "No features extracted",
                })
            else:
                file_features_df.to_csv(
                    FEATURES_PATH,
                    mode="a",
                    header=not wrote_header,
                    index=False,
                )
                wrote_header = True
                processed_files += 1

        except Exception as exc:
            skipped_records.append({
                "file_path": str(file_path),
                "reason": f"Unhandled error: {exc}",
            })
        finally:
            gc.collect()

    skipped_df = pd.DataFrame(skipped_records, columns=["file_path", "reason"])
    skipped_df.to_csv(SKIPPED_FILES_PATH, index=False)

    if FEATURES_PATH.exists():
        features_df_for_count = pd.read_csv(FEATURES_PATH, nrows=5)
        feature_rows_created = max(0, sum(1 for _ in open(FEATURES_PATH, "r", encoding="utf-8")) - 1)
        metadata_cols = {"source_file", "window_index", "global_order_index"}
        EXTRACTION_SUMMARY["final_feature_column_count"] = int(len([c for c in features_df_for_count.columns if c not in metadata_cols]))
    else:
        feature_rows_created = 0

    EXTRACTION_SUMMARY["number_of_files_processed"] = int(processed_files)
    EXTRACTION_SUMMARY["number_of_skipped_files"] = int(len(skipped_df))
    EXTRACTION_SUMMARY["total_extracted_windows"] = int(feature_rows_created)

    with open(FEATURE_EXTRACTION_REPORT_PATH, "w", encoding="utf-8") as report_file:
        report_file.write("Feature Extraction Report\n")
        report_file.write("=========================\n\n")
        for key, value in EXTRACTION_SUMMARY.items():
            report_file.write(f"{key}: {value}\n")

    print(f"Files selected (MAX_FILES): {len(files_to_process)}")
    print(f"Files processed successfully: {processed_files}")
    print(f"Feature rows created: {feature_rows_created}")
    print(f"Skipped files: {len(skipped_df)}")
    print(f"Feature file saved to: {FEATURES_PATH}")
    print(f"Skipped file report saved to: {SKIPPED_FILES_PATH}")
    print(f"Feature extraction report saved to: {FEATURE_EXTRACTION_REPORT_PATH}")

if SKIPPED_FILES_PATH.exists() and SKIPPED_FILES_PATH.stat().st_size > 0:
    display(pd.read_csv(SKIPPED_FILES_PATH).head())
else:
    display(pd.DataFrame(columns=["file_path", "reason"]))
print("\nExtraction summary:")
print(json.dumps(EXTRACTION_SUMMARY, indent=2))

Feature extraction:   0%|          | 0/129 [00:00<?, ?it/s]

Files selected (MAX_FILES): 129
Files processed successfully: 129
Feature rows created: 125775
Skipped files: 0
Feature file saved to: /kaggle/working/processed/bearing_features.csv
Skipped file report saved to: /kaggle/working/reports/skipped_files.csv


,file_path,reason


## 6. Pseudo Label Creation

In [18]:
if not FEATURES_PATH.exists():
    raise FileNotFoundError(f"Feature file not found: {FEATURES_PATH}")

features_df = pd.read_csv(FEATURES_PATH)
if features_df.empty:
    raise ValueError("Feature file is empty. Check skipped file report and raw inputs.")

features_df = features_df.sort_values("global_order_index").reset_index(drop=True)

n_rows = len(features_df)
normal_end = int(n_rows * 0.60)
warning_end = int(n_rows * 0.85)

labels = np.array(["critical"] * n_rows, dtype=object)
labels[:normal_end] = "normal"
labels[normal_end:warning_end] = "warning"

features_df["health_label"] = labels
features_df.to_csv(LABELED_FEATURES_PATH, index=False)

label_distribution = features_df["health_label"].value_counts()
print("Label distribution:")
print(label_distribution)
print(f"Saved labeled dataset to: {LABELED_FEATURES_PATH}")

Label distribution:
health_label
normal      75465
warning     31443
critical    18867
Name: count, dtype: int64
Saved labeled dataset to: /kaggle/working/processed/bearing_features_labeled.csv


## 7. Exploratory Plots

In [ ]:
plot_df = pd.read_csv(LABELED_FEATURES_PATH)

saved_plots = []

if "vibration_resultant_rms" in plot_df.columns:
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(plot_df["global_order_index"], plot_df["vibration_resultant_rms"], linewidth=1.2, color="tab:blue")
    ax.set_title("Vibration Resultant RMS Over Time")
    ax.set_xlabel("Global Window Index")
    ax.set_ylabel("vibration_resultant_rms")
    rms_plot_path = REPORT_DIR / "vibration_resultant_rms_over_time.png"
    fig.savefig(rms_plot_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    saved_plots.append(str(rms_plot_path))

if "vibration_resultant_kurtosis" in plot_df.columns:
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(plot_df["global_order_index"], plot_df["vibration_resultant_kurtosis"], linewidth=1.2, color="tab:orange")
    ax.set_title("Vibration Resultant Kurtosis Over Time")
    ax.set_xlabel("Global Window Index")
    ax.set_ylabel("vibration_resultant_kurtosis")
    kurt_plot_path = REPORT_DIR / "vibration_resultant_kurtosis_over_time.png"
    fig.savefig(kurt_plot_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    saved_plots.append(str(kurt_plot_path))

if "temperature_bearing_mean" in plot_df.columns and plot_df["temperature_bearing_mean"].notna().any():
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(plot_df["global_order_index"], plot_df["temperature_bearing_mean"], linewidth=1.2, color="tab:red")
    ax.set_title("Bearing Temperature Mean Over Time")
    ax.set_xlabel("Global Window Index")
    ax.set_ylabel("temperature_bearing_mean")
    temp_bearing_plot_path = REPORT_DIR / "temperature_bearing_mean_over_time.png"
    fig.savefig(temp_bearing_plot_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    saved_plots.append(str(temp_bearing_plot_path))

if "temperature_difference_mean" in plot_df.columns and plot_df["temperature_difference_mean"].notna().any():
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(plot_df["global_order_index"], plot_df["temperature_difference_mean"], linewidth=1.2, color="tab:purple")
    ax.set_title("Temperature Difference Mean Over Time")
    ax.set_xlabel("Global Window Index")
    ax.set_ylabel("temperature_difference_mean")
    temp_diff_plot_path = REPORT_DIR / "temperature_difference_mean_over_time.png"
    fig.savefig(temp_diff_plot_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    saved_plots.append(str(temp_diff_plot_path))

label_counts = plot_df["health_label"].value_counts().reindex(["normal", "warning", "critical"], fill_value=0)
fig, ax = plt.subplots(figsize=(6, 4))
label_counts.plot(kind="bar", ax=ax, color=["tab:green", "tab:orange", "tab:red"])
ax.set_title("Health Label Distribution")
ax.set_xlabel("Health Label")
ax.set_ylabel("Count")
dist_plot_path = REPORT_DIR / "health_label_distribution.png"
fig.savefig(dist_plot_path, dpi=150, bbox_inches="tight")
plt.close(fig)
saved_plots.append(str(dist_plot_path))

print("Saved plots:")
for plot_path in saved_plots:
    print(plot_path)

Saved plots:
/kaggle/working/reports/rms_over_time.png
/kaggle/working/reports/kurtosis_over_time.png
/kaggle/working/reports/health_label_distribution.png


## 8. Train Basic Isolation Forest Model

In [20]:
model_df = pd.read_csv(LABELED_FEATURES_PATH)

metadata_columns = {"source_file", "window_index", "global_order_index", "health_label"}
numeric_columns = model_df.select_dtypes(include=[np.number]).columns.tolist()
feature_columns = [col for col in numeric_columns if col not in metadata_columns]

if len(feature_columns) == 0:
    raise ValueError("No usable numeric feature columns found for model training.")

X_all = model_df[feature_columns].copy()
y_all = model_df["health_label"].astype(str)

normal_mask = y_all == "normal"
if normal_mask.sum() < 2:
    raise ValueError("Not enough 'normal' samples to train Isolation Forest.")

model_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("isolationforest", IsolationForest(contamination=0.1, random_state=RANDOM_STATE))
    ]
)

model_pipeline.fit(X_all.loc[normal_mask])

risk_scores = -model_pipeline.decision_function(X_all)

risk_threshold_low = float(np.quantile(risk_scores, 0.60))
risk_threshold_high = float(np.quantile(risk_scores, 0.85))

def risk_to_status(score, low_thr, high_thr):
    if score <= low_thr:
        return "normal"
    if score <= high_thr:
        return "warning"
    return "critical"

model_df["anomaly_score"] = risk_scores
model_df["predicted_status"] = [
    risk_to_status(score, risk_threshold_low, risk_threshold_high)
    for score in risk_scores
]

print("Training complete.")
print(f"Feature columns used: {len(feature_columns)}")
print(f"Risk thresholds -> low: {risk_threshold_low:.6f}, high: {risk_threshold_high:.6f}")
display(model_df[["anomaly_score", "predicted_status"]].head())

/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['temperature_mean' 'temperature_min' 'temperature_max' 'temperature_std'
 'temperature_trend']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['temperature_mean' 'temperature_min' 'temperature_max' 'temperature_std'
 'temperature_trend']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


Training complete.
Feature columns used: 18
Risk thresholds -> low: -0.053267, high: -0.003188


,anomaly_score,predicted_status
0,-0.076104,normal
1,-0.008527,warning
2,-0.095305,normal
3,-0.088732,normal
4,-0.027749,warning


## 9. Evaluation

In [ ]:
label_order = ["normal", "warning", "critical"]

y_true = model_df["health_label"].astype(str)
y_pred = model_df["predicted_status"].astype(str)

report_text = classification_report(
    y_true,
    y_pred,
    labels=label_order,
    digits=4,
    zero_division=0,
)

cm = confusion_matrix(y_true, y_pred, labels=label_order)
accuracy = accuracy_score(y_true, y_pred)
model_accuracy = float(accuracy)

print("Classification Report:")
print(report_text)
print(f"Accuracy: {accuracy:.4f}")

cm_df = pd.DataFrame(cm, index=label_order, columns=label_order)
cm_df.to_csv(CONFUSION_MATRIX_PATH, index=True)

with open(EVAL_REPORT_PATH, "w", encoding="utf-8") as f:
    f.write("Basic Bearing Predictive Maintenance Model Evaluation\n")
    f.write("===============================================\n\n")

    f.write("Extraction Summary:\n")
    for key, value in EXTRACTION_SUMMARY.items():
        f.write(f"- {key}: {value}\n")
    f.write("\n")

    f.write(f"Final model feature column count: {len(feature_columns)}\n\n")

    f.write("Classification Report:\n")
    f.write(report_text)
    f.write("\n")

    f.write(f"Accuracy: {accuracy:.6f}\n\n")

    f.write("Confusion Matrix (rows=true, cols=pred):\n")
    f.write(cm_df.to_string())
    f.write("\n")

print(f"Saved evaluation report to: {EVAL_REPORT_PATH}")
print(f"Saved confusion matrix to: {CONFUSION_MATRIX_PATH}")

Classification Report:
              precision    recall  f1-score   support

      normal     0.6547    0.6547    0.6547     75465
     warning     0.3235    0.3235    0.3235     31443
    critical     0.2494    0.2494    0.2494     18867

    accuracy                         0.5111    125775
   macro avg     0.4092    0.4092    0.4092    125775
weighted avg     0.5111    0.5111    0.5111    125775

Accuracy: 0.5111
Saved evaluation report to: /kaggle/working/reports/model_evaluation_report.txt
Saved confusion matrix to: /kaggle/working/reports/confusion_matrix.csv


## 10. Save Model

In [22]:
joblib.dump(model_pipeline, MODEL_PATH)

with open(FEATURE_COLUMNS_PATH, "w", encoding="utf-8") as f:
    json.dump(feature_columns, f, indent=2)

threshold_payload = {
    "risk_score_quantile_60": risk_threshold_low,
    "risk_score_quantile_85": risk_threshold_high
}

with open(THRESHOLDS_PATH, "w", encoding="utf-8") as f:
    json.dump(threshold_payload, f, indent=2)

print(f"Saved model to: {MODEL_PATH}")
print(f"Saved feature column names to: {FEATURE_COLUMNS_PATH}")
print(f"Saved anomaly thresholds to: {THRESHOLDS_PATH}")

Saved model to: /kaggle/working/models/vibration_isolation_forest.pkl
Saved feature column names to: /kaggle/working/models/feature_columns.json
Saved anomaly thresholds to: /kaggle/working/models/anomaly_thresholds.json


## 11. Test Sample Prediction

In [23]:
loaded_model = joblib.load(MODEL_PATH)

with open(FEATURE_COLUMNS_PATH, "r", encoding="utf-8") as f:
    saved_feature_columns = json.load(f)

with open(THRESHOLDS_PATH, "r", encoding="utf-8") as f:
    saved_thresholds = json.load(f)

sample_row = pd.read_csv(LABELED_FEATURES_PATH, nrows=1)
sample_features = sample_row[saved_feature_columns]

sample_risk_score = float(-loaded_model.decision_function(sample_features)[0])

low_thr = float(saved_thresholds["risk_score_quantile_60"])
high_thr = float(saved_thresholds["risk_score_quantile_85"])

if sample_risk_score <= low_thr:
    sample_status = "normal"
elif sample_risk_score <= high_thr:
    sample_status = "warning"
else:
    sample_status = "critical"

sample_prediction = {
    "device_id": "demo_device_001",
    "asset_id": "bearing_motor_001",
    "model_name": "vibration_isolation_forest",
    "model_version": "v1",
    "predicted_status": sample_status,
    "anomaly_score": round(sample_risk_score, 6)
}

print(json.dumps(sample_prediction, indent=2))

{
  "device_id": "demo_device_001",
  "asset_id": "bearing_motor_001",
  "model_name": "vibration_isolation_forest",
  "model_version": "v1",
  "predicted_status": "normal",
  "anomaly_score": -0.076104
}


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['temperature_mean' 'temperature_min' 'temperature_max' 'temperature_std'
 'temperature_trend']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


## 12. Final Output Summary

In [ ]:
final_file_index_df = pd.read_csv(FILE_INDEX_PATH) if FILE_INDEX_PATH.exists() else pd.DataFrame()
final_features_df = pd.read_csv(FEATURES_PATH) if FEATURES_PATH.exists() else pd.DataFrame()
final_labeled_df = pd.read_csv(LABELED_FEATURES_PATH) if LABELED_FEATURES_PATH.exists() else pd.DataFrame()

if not final_labeled_df.empty and "health_label" in final_labeled_df.columns:
    final_label_distribution = final_labeled_df["health_label"].value_counts().to_dict()
else:
    final_label_distribution = {}

confusion_matrix_values = cm.tolist() if "cm" in globals() else None

final_summary = {
    "number_of_files_discovered": int(len(final_file_index_df)),
    "number_of_files_processed": int(EXTRACTION_SUMMARY.get("number_of_files_processed", 0)),
    "number_of_skipped_files": int(EXTRACTION_SUMMARY.get("number_of_skipped_files", 0)),
    "total_extracted_windows": int(EXTRACTION_SUMMARY.get("total_extracted_windows", 0)),
    "vibration_x_detected": bool(EXTRACTION_SUMMARY.get("vibration_x_detected", False)),
    "vibration_y_detected": bool(EXTRACTION_SUMMARY.get("vibration_y_detected", False)),
    "temperature_bearing_detected": bool(EXTRACTION_SUMMARY.get("temperature_bearing_detected", False)),
    "temperature_atmospheric_detected": bool(EXTRACTION_SUMMARY.get("temperature_atmospheric_detected", False)),
    "final_feature_column_count": int(EXTRACTION_SUMMARY.get("final_feature_column_count", 0)),
    "label_distribution": final_label_distribution,
    "model_accuracy": float(model_accuracy) if "model_accuracy" in globals() else None,
    "confusion_matrix": confusion_matrix_values,
    "confusion_matrix_path": str(CONFUSION_MATRIX_PATH),
    "feature_extraction_report_path": str(FEATURE_EXTRACTION_REPORT_PATH),
    "model_saved_path": str(MODEL_PATH),
    "feature_csv_path": str(FEATURES_PATH),
    "evaluation_report_path": str(EVAL_REPORT_PATH),
}

# Keep a single report file that includes extraction and model-level summary metrics.
with open(FEATURE_EXTRACTION_REPORT_PATH, "w", encoding="utf-8") as report_file:
    report_file.write("Feature and Model Summary Report\n")
    report_file.write("================================\n\n")
    for key, value in final_summary.items():
        report_file.write(f"{key}: {value}\n")

print("Final Output Summary")
print(json.dumps(final_summary, indent=2))

expected_outputs = [
    FILE_INDEX_PATH,
    FEATURES_PATH,
    LABELED_FEATURES_PATH,
    MODEL_PATH,
    FEATURE_COLUMNS_PATH,
    THRESHOLDS_PATH,
    FEATURE_EXTRACTION_REPORT_PATH,
    EVAL_REPORT_PATH,
    CONFUSION_MATRIX_PATH,
    SKIPPED_FILES_PATH,
]

print("\nExpected output files")
for output_path in expected_outputs:
    print(f"{output_path} -> exists: {output_path.exists()}")

Final Output Summary
{
  "number_of_files_discovered": 129,
  "number_of_files_processed": 129,
  "number_of_feature_rows_created": 125775,
  "label_distribution": {
    "normal": 75465,
    "warning": 31443,
    "critical": 18867
  },
  "model_saved_path": "/kaggle/working/models/vibration_isolation_forest.pkl",
  "feature_csv_path": "/kaggle/working/processed/bearing_features.csv",
  "evaluation_report_path": "/kaggle/working/reports/model_evaluation_report.txt"
}

Expected output files
/kaggle/working/processed/file_index.csv -> exists: True
/kaggle/working/processed/bearing_features.csv -> exists: True
/kaggle/working/processed/bearing_features_labeled.csv -> exists: True
/kaggle/working/models/vibration_isolation_forest.pkl -> exists: True
/kaggle/working/models/feature_columns.json -> exists: True
/kaggle/working/models/anomaly_thresholds.json -> exists: True
/kaggle/working/reports/model_evaluation_report.txt -> exists: True
/kaggle/working/reports/confusion_matrix.csv -> exists